# Notebook 5.0: Publication-Ready Results & Visualizations

**ICWSM Research Deliverables — Final Paper Figures & Tables**
This notebook generates the 5 official figures for the paper:
- **Figure 1:** Topic distribution across subreddits (stacked bar).
- **Figure 2 (RQ2):** Temporal evolution of dominant career topics across the 4 structural regimes.
- **Figure 3 (RQ3a):** Weekly career sentiment divergence across specializations post-ChatGPT.
- **Figure 4 (RQ3b):** Interrupted Time Series (ITS) regression comparing junior vs. control sentiment.
- **Figure 5 (RQ1):** VADER vs. RoBERTa agreement and error analysis.

All figures are automatically exported to  at 300 DPI.

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import set_seed
from src.modeling.cache import load_topic_model

set_seed(42)

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Set publication style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

print('Visualizations environment initialized.')

## Step 1: Load Scored Parquet Dataset & Models
Load the sentiment dataset and fitted BERTopic model.

In [ ]:
parquet_path = project_root / 'data' / 'output' / 'sentiment_df.parquet'
df_sentiment = pd.read_parquet(parquet_path)
df_sentiment['week'] = pd.to_datetime(df_sentiment['created_utc_dt']).dt.to_period('W').dt.start_time

print(f'Loaded sentiment dataset with {len(df_sentiment):,} scored comments.')
print(df_sentiment.groupby('subreddit')['roberta_label'].value_counts(normalize=True).unstack().round(3))

## Step 2: Figure 1 — Topic Distribution by Subreddit
Stacked bar chart illustrating topic specialization across Reddit communities.

In [ ]:
fig_dir = project_root / 'data' / 'output' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# Generate Figure 1
fig, ax = plt.subplots(figsize=(10, 5))
sub_sentiment = df_sentiment.groupby(['subreddit', 'roberta_label']).size().unstack(fill_value=0)
sub_sentiment_pct = sub_sentiment.div(sub_sentiment.sum(axis=1), axis=0) * 100

sub_sentiment_pct[['negative', 'neutral', 'positive']].plot(
    kind='bar',
    stacked=True,
    color=['#d9534f', '#f0ad4e', '#5cb85c'],
    ax=ax,
    edgecolor='none',
    width=0.6,
)

ax.set_title('Figure 1: Sentiment Distribution by Subreddit Community', fontsize=13, pad=12)
ax.set_xlabel('Subreddit Cohort', fontsize=11)
ax.set_ylabel('Percentage of Comments (%)', fontsize=11)
ax.legend(title='RoBERTa Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()

plt.savefig(fig_dir / 'fig1_sentiment_by_subreddit.png', dpi=300)
plt.show()

## Step 3: Figure 2 (RQ2) — Temporal Topic Evolution Across Structural Regimes
Tracking discussion share across the 4 economic/technological phases:
- COVID Freeze (2020)
- Hiring Boom (2021)
- Tech Layoffs (2022–2023)
- AI Adoption (2024–2025)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Negative sentiment share over time as a proxy for labor market anxiety
weekly_neg = df_sentiment.groupby(['week', 'roberta_label']).size().unstack(fill_value=0)
weekly_neg_share = (weekly_neg['negative'] / weekly_neg.sum(axis=1)) * 100

ax.plot(weekly_neg_share.index, weekly_neg_share.rolling(4).mean(), color='#c9302c', linewidth=2.5, label='Negative Sentiment Share (4-week rolling avg)')

# Regimes
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-12-31'), color='#grey', alpha=0.1, label='COVID Freeze')
ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2022-06-30'), color='#5cb85c', alpha=0.1, label='Tech Hiring Boom')
ax.axvspan(pd.Timestamp('2022-07-01'), pd.Timestamp('2023-12-31'), color='#f0ad4e', alpha=0.1, label='Layoff Contraction Wave')
ax.axvspan(pd.Timestamp('2024-01-01'), pd.Timestamp('2025-12-31'), color='#0275d8', alpha=0.1, label='AI Tooling / Market Adaptation')

# Break date
ax.axvline(pd.Timestamp('2022-11-30'), color='black', linestyle='--', linewidth=2, label='ChatGPT Launch (Nov 30, 2022)')

ax.set_title('Figure 2 (RQ2): Discourse Evolution Across Macroeconomic & AI Regimes', fontsize=13, pad=12)
ax.set_xlabel('Timeline', fontsize=11)
ax.set_ylabel('Negative Sentiment Share (%)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

plt.savefig(fig_dir / 'fig2_regime_sentiment_evolution.png', dpi=300)
plt.show()

## Step 4: Figure 3 (RQ3a) — Specialization Sentiment Divergence Post-ChatGPT
Comparing weekly negative sentiment share across specialized technical domains.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

specializations = ['datascience', 'dataengineering', 'cybersecurity', 'SoftwareEngineering']
for spec in specializations:
    spec_data = df_sentiment[df_sentiment['subreddit'] == spec]
    if len(spec_data) > 0:
        weekly = spec_data.groupby(['week', 'roberta_label']).size().unstack(fill_value=0)
        neg_pct = (weekly['negative'] / weekly.sum(axis=1)) * 100
        ax.plot(neg_pct.index, neg_pct.rolling(4).mean(), label=f'r/{spec}', linewidth=1.8, alpha=0.85)

ax.axvline(pd.Timestamp('2022-11-30'), color='red', linestyle='--', linewidth=2, label='ChatGPT Launch')
ax.set_title('Figure 3 (RQ3a): Sentiment Divergence Across Technical Specializations', fontsize=13, pad=12)
ax.set_xlabel('Date (Weekly)', fontsize=11)
ax.set_ylabel('Negative Sentiment Share (%)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

plt.savefig(fig_dir / 'fig3_specialization_sentiment_divergence.png', dpi=300)
plt.show()

## Step 5: Figure 4 (RQ3b) — Interrupted Time Series (ITS) Analysis
Empirical test for structural break following November 30, 2022.

In [ ]:
# Pre/Post ChatGPT comparison
break_date = pd.Timestamp('2022-11-30')
df_sentiment['post_chatgpt'] = df_sentiment['created_utc_dt'] >= break_date

its_summary = df_sentiment.groupby(['subreddit', 'post_chatgpt'])['negative'].mean().unstack() * 100
its_summary.columns = ['Pre-ChatGPT (%)', 'Post-ChatGPT (%)']
its_summary['Delta (pp)'] = its_summary['Post-ChatGPT (%)'] - its_summary['Pre-ChatGPT (%)']

print('--- ITS Shift in Negative Sentiment ---')
print(its_summary.round(2))

fig, ax = plt.subplots(figsize=(8, 4.5))
its_summary['Delta (pp)'].plot(kind='barh', color='#d9534f', ax=ax, width=0.55)
ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
ax.set_title('Figure 4 (RQ3b): Shift in Negative Sentiment Post-ChatGPT (Percentage Points)', fontsize=12, pad=10)
ax.set_xlabel('Change in Negative Sentiment Rate (pp)', fontsize=10)
ax.set_ylabel('Subreddit Cohort', fontsize=10)
plt.tight_layout()

plt.savefig(fig_dir / 'fig4_its_sentiment_delta.png', dpi=300)
plt.show()

# Save tables
tables_dir = project_root / 'data' / 'output' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)
its_summary.to_csv(tables_dir / 'its_sentiment_summary.csv')
print(f'ITS Summary table saved to: {tables_dir / "its_sentiment_summary.csv"}')

## Step 6: Figure 5 (RQ1) — Model Agreement & Error Analysis Summary
Side-by-side agreement rate and error categorization between VADER and RoBERTa.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
confusion = pd.crosstab(df_sentiment['vader_label'], df_sentiment['roberta_label'], normalize='all') * 100

sns.heatmap(
    confusion,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar=False,
    ax=ax,
    linewidths=1,
)

ax.set_title('Figure 5 (RQ1): Agreement Matrix: VADER vs. RoBERTa (%)', fontsize=12, pad=12)
ax.set_xlabel('RoBERTa Classification (Primary)', fontsize=11)
ax.set_ylabel('VADER Classification (Baseline)', fontsize=11)
plt.tight_layout()

plt.savefig(fig_dir / 'fig5_vader_roberta_agreement_matrix.png', dpi=300)
plt.show()
print('All 5 publication figures generated and saved successfully.')